# Closed-Source Models -- African Language Confusion Sweep

Clean, systematic version of the exploratory `Closed_Source_Models.ipynb` blueprint, for the
closed-source models (GPT, Claude, Gemini, Llama) reachable through the CMU AI Gateway.
Runs every model in `MODELS` over the full consolidated prompt set built by
`Build_Prompt_Dataset.ipynb` (`prompts/all_prompts.csv`), logging to
`outputs/{model_key}.csv` in the same `id, model, completion, task, source, language`
schema the language-confusion repo's `compute_metrics.py` expects.

**Change from the blueprint's ad hoc cells:** each prompt is sent as an independent
single-turn message rather than appended to a growing conversation -- see
`model_clients.py` (shared with `Open_Source_Models.ipynb`) for why, and for the
sweep/retry/resume logic used here.

### Setup

Run once, from the `african-language-confusion/` folder, in a virtual environment (keeps
these dependencies out of your global Python install):

```bash
python -m venv venv
venv\Scriptsctivate      # Windows
source venv/bin/activate    # macOS/Linux
pip install -r requirements.txt
```

Then pick `venv` as this notebook's kernel before running the cells below.


In [1]:
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

import model_clients as mc

CMU_OPENAI_API_KEY = mc.get_api_key("CMU_OPENAI_API_KEY")
CMU_BASE_URL = "https://ai-gateway.andrew.cmu.edu/"


### Models

CMU AI Gateway model ids -- add/remove entries here to change what the sweep covers.

In [2]:
MODELS = {
    "gpt": "gpt-5.5",
    "claude": "us.anthropic.claude-opus-4-8",
    "gemini": "gemini/gemini-3.5-flash",
}


### Chat helper

One independent, single-turn call per prompt (see the design note above).

In [3]:
def make_call_fn(model_id, temperature=0.5, max_tokens=1000):
    client = ChatOpenAI(
        model=model_id,
        base_url=CMU_BASE_URL,
        api_key=CMU_OPENAI_API_KEY,
        temperature=temperature,
        max_tokens=max_tokens,
    )

    def call_fn(prompt):
        response = client.invoke([HumanMessage(content=prompt)])
        return response.content
    return call_fn


### Load prompts

In [4]:
all_prompts = pd.read_csv("prompts/all_prompts.csv")
all_prompts.shape


(22431, 5)

### Quick check on a single language before running the full sweep

In [5]:
prompts_preview = mc.get_prompts(all_prompts, language="igbo", source="aya", n=5)
call_fn = make_call_fn(MODELS["gpt"])
for _, row in prompts_preview.iterrows():
    print(f"PROMPT: {row['prompt']}")
    print(f"REPLY:  {call_fn(row['prompt'])}")
    print()


PROMPT: Kọwaa nkenke ụdị ejije e nwere?
REPLY:  Ụdị ejije e nwere gụnyere ndị a:

1. **Ejije ọchị (Comedy)** – Ejije na-eme ka ndị na-ekiri ya chịa ọchị; ọ na-enwekarị njedebe ọma.

2. **Ejije ọdachi (Tragedy)** – Ejije jupụtara n’ahụhụ, nsogbu ma ọ bụ ọnwụ; njedebe ya na-abụkarị ihe mwute.

3. **Ejije ọchị na mwute (Tragicomedy)** – Ejije jikọtara ihe ọchị na ihe mwute n’otu akụkọ.

4. **Ejije egwu/abụ (Musical drama)** – Ejije a na-eji egwu, abụ na ịgba egwu akọwa akụkọ ya.

5. **Ejije akụkọ ihe mere eme (Historical drama)** – Ejije dabere n’ihe mere n’oge gara aga ma ọ bụ ndụ ndị ama ama.

6. **Ejije nkuzi (Didactic drama)** – Ejije e ji akụzi omume ọma, omenala, ma ọ bụ ihe mmụta.

7. **Ejije nkatọ (Satirical drama)** – Ejije na-akọcha ma ọ bụ na-akwa emo àgwà ọjọọ dị n’obodo iji mee ka a gbanwee ya.

PROMPT: Gini bu azịza gwam gwam gwam(agwụgwa) a

Gwa m oti nwata n'anya nne ya
REPLY:  Azịza agwụgwa ahụ bụ: **anwụrụ ọkụ**.

**Nkọwa:** Anwụrụ ọkụ nwere ike “iti” ma ọ bụ gbaa nwata 

### Full sweep

Runs every model in `MODELS` over every prompt in `all_prompts`, writing/resuming
`outputs/{model_key}.csv`. Scope it down first with
`mc.get_prompts(all_prompts, language=[...], source=[...], task=...)` if you don't want a
full run yet. `delay` adds a small pause between calls to stay under rate limits.


In [ ]:
output_paths = mc.run_sweep(
    MODELS,
    make_call_fn,
    all_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths


### Per-dataset sweeps

Same sweep as above, scoped to one prompt dataset (`source`) at a time. Useful for running/
resuming a single dataset in isolation (e.g. to prioritize it, or re-run after a dataset-
specific fix) instead of the full `all_prompts` set. These write into the same
`outputs/{model_key}.csv` files as the full sweep -- `run_benchmark` skips ids it has
already logged, so running a per-dataset cell and the full sweep cell against the same
`outputs/` directory is safe either order.

#### aya

In [ ]:
aya_prompts = mc.get_prompts(all_prompts, source="aya")
output_paths_aya = mc.run_sweep(
    MODELS,
    make_call_fn,
    aya_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_aya

#### afriqa

In [ ]:
afriqa_prompts = mc.get_prompts(all_prompts, source="afriqa")
output_paths_afriqa = mc.run_sweep(
    MODELS,
    make_call_fn,
    afriqa_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_afriqa

#### polywrite

In [ ]:
polywrite_prompts = mc.get_prompts(all_prompts, source="polywrite")
output_paths_polywrite = mc.run_sweep(
    MODELS,
    make_call_fn,
    polywrite_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_polywrite

#### dolly

In [ ]:
dolly_prompts = mc.get_prompts(all_prompts, source="dolly")
output_paths_dolly = mc.run_sweep(
    MODELS,
    make_call_fn,
    dolly_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_dolly

#### sharegpt

In [ ]:
sharegpt_prompts = mc.get_prompts(all_prompts, source="sharegpt")
output_paths_sharegpt = mc.run_sweep(
    MODELS,
    make_call_fn,
    sharegpt_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_sharegpt

#### okapi

In [ ]:
okapi_prompts = mc.get_prompts(all_prompts, source="okapi")
output_paths_okapi = mc.run_sweep(
    MODELS,
    make_call_fn,
    okapi_prompts,
    output_dir="outputs",
    delay=1.0,
)
output_paths_okapi